# Day 4-02｜鎖定投籃者、建立骨架與關節角度
> Python 籃球運動資料分析課程  
> 這一節我會帶你們把投籃候選模型與 YOLO pose 接在一起，讓每個 frame 都知道投籃者是誰、
> 人體 keypoints 在哪裡，以及手肘、肩膀與膝蓋角度是多少。

## 你們會完成什麼
- 使用 `shot_detection.pt` 取得逐 frame 的 `shot_score` 與投籃者候選框。
- 使用 `yolov8n-pose.pt` 找出人體 keypoints，並在多人畫面中維持同一位投籃者。
- 把左右側關節點換算成 2D 關節角度，輸出 CSV、曲線圖與 overlay 影片。
- 用肉眼確認骨架沒有跳到其他人，並理解 `wrist` 標記只是當下手腕位置，不是球軌跡。

## 怎樣算完成
- 骨架在投籃前後大致黏在同一位投籃者身上。
- `shot_score` 高峰和肉眼看到的投籃動作大致重疊。
- 角度曲線沒有大量單幀跳動，overlay 的手腕與肢體位置可信。

## 本節產出
- `assets/results/d4_02_pose_angles.csv`
- `assets/results/d4_02_pose_angle_plot.png`
- `assets/results/d4_02_pose_overlay_preview.mp4`


## Setup｜確認模型與結果資料夾

Colab 使用者請先執行 `init_colab.ipynb`，並確認 runtime 是 T4 GPU。下一個程式格只負責掛載 Drive、
找到課程根目錄與檢查模型，所以不逐行展開。執行後，`results directory` 必須位於
`/content/drive/MyDrive/basketball_hackathon/course/assets/results`。

本節預設分析前 180 frames，而且 `POSE_STRIDE = 1`。快速測試可以暫時改成 `2`，但正式做 release
分析前要改回 `1`，否則時間解析度會少一半。


In [ ]:
from pathlib import Path
import subprocess
import sys

DRIVE_MOUNTED = False
if "google.colab" in sys.modules:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("目前這個 Colab runtime 不支援 Drive 掛載，改用 /content 本機路徑。")

COURSE_ROOT_HINT = (
    Path("/content/drive/MyDrive/basketball_hackathon/course")
    if DRIVE_MOUNTED
    else next(
        (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (p / "src" / "course_setup.py").exists()),
        Path("/content/basketball_hackathon/course"),
    )
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo, prepare_day4_workspace  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT, mount_drive=False)
DAY4_MODELS, RESULTS = prepare_day4_workspace(COURSE_ROOT)
print("Day 4 models:", {name: str(path) for name, path in DAY4_MODELS.items()})
print("results directory:", RESULTS)


## Step 1｜讀取 Day 4-01 使用的影片

這個程式格只選擇輸入影片並載入後面會用到的顯示、畫圖與裝置工具。`pick_first_converted_video`
會優先讀取 `assets/converted/` 內排序最前面的影片；如果 Day 4-01 產生了
`student_video.mp4`，三份 notebook 都會使用它。

請確認印出的 `using video` 和 Day 4-01 完全相同。影片不同時，即使 CSV 能讀取，frame 也不會對齊。


In [ ]:
from src.plot_utils import plot_angle_series
from src.shooting_utils import render_pose_overlay_video
from src.video_utils import display_video_in_notebook, list_videos, pick_first_converted_video
from src.yolo_utils import preferred_inference_device

converted_videos = list_videos(COURSE_ROOT / "assets" / "converted")
video_path = pick_first_converted_video(COURSE_ROOT)

print("converted videos:", [path.name for path in converted_videos])
print("using video:", video_path)


## Step 2｜同時執行投籃候選與人體姿態推論

這是本節的核心模型格：

1. `shot_detection.pt` 對每個 frame 產生 `shot_score` 和人物候選框。
2. `yolov8n-pose.pt` 找出畫面中每個人的 17 個 COCO keypoints。
3. 有明確 shot box 時，程式選擇與它最相符的人；訊號暫時變弱時，則根據前一個人物位置維持身分。
4. 左右肩、手肘、手腕、髖、膝與腳踝會換算成雙側 2D 角度。

`POSE_SIDE = "right"` 只決定簡化欄位 `elbow_angle`、`shoulder_angle`、`knee_angle` 顯示哪一側；
CSV 仍保留左右兩側。執行後請檢查 `rows`、`shot_score`、人物框與 keypoint 欄位是否都有資料。


In [ ]:
from src.shot_event_utils import extract_yolo_shot_pose_sequence

POSE_SIDE = "right"
POSE_STRIDE = 1
POSE_MAX_FRAMES = 180
DEVICE = preferred_inference_device()
pose_model_path = DAY4_MODELS["yolo_pose"]
shot_model_path = DAY4_MODELS["shot_detection"]

# 每一列代表一個分析 frame，包含 shot score、投籃者框、keypoints 與左右側角度。
pose_df = extract_yolo_shot_pose_sequence(
    video_path,
    pose_model_path=pose_model_path,
    shot_model_path=shot_model_path,
    stride=POSE_STRIDE,
    side=POSE_SIDE,
    max_frames=POSE_MAX_FRAMES,
    device=DEVICE,
)

print("pose model:", pose_model_path)
print("shot model:", shot_model_path)
print("inference device:", DEVICE)
print("rows:", len(pose_df))
pose_df.head()


## Step 3｜儲存逐 frame 資料並畫角度曲線

CSV 是 Day 4-03 的正式輸入；曲線圖是快速檢查工具。`to_csv` 會保留每個 frame 的 shot score、
人物框、keypoints 和角度，`plot_angle_series` 則把手肘、膝蓋、肩膀角度畫成時間序列。

請先看投籃前的下蹲、抬球與出手後伸展是否形成連續變化。單一 frame 突然跳高或掉低時，我會先
回頭檢查關節點；2D pose 會受遮擋、側身與模糊影響，數值異常不一定代表動作異常。


In [ ]:
pose_csv = RESULTS / "d4_02_pose_angles.csv"
angle_plot_png = RESULTS / "d4_02_pose_angle_plot.png"

# CSV 留給 Day 4-03 讀取，圖檔留給我們快速檢查時間序列。
pose_df.to_csv(pose_csv, index=False)
plot_angle_series(
    pose_df,
    ["elbow_angle", "knee_angle", "shoulder_angle"],
    output_path=angle_plot_png,
)

print("saved:", pose_csv)
print("saved:", angle_plot_png)


## Step 4｜把骨架疊回影片做人工驗證

這個格子會把 `pose_df` 的骨架畫回原影片。白色骨架應留在同一位投籃者身上；粉紅色 `wrist`
只標示當下手腕位置，不會畫手腕歷史線，也不是籃球軌跡。

如果人物選錯，後面的 release frame 和角度即使有數字也沒有分析價值。請先把 frame 記下來，
不要直接解讀 Day 4-03 的結果。


In [ ]:
pose_overlay_mp4 = RESULTS / "d4_02_pose_overlay_preview.mp4"

# highlight_joint 只畫目前 frame 的手腕點；人物身分由前面的 pose_df 決定。
render_pose_overlay_video(
    video_path,
    pose_df,
    pose_overlay_mp4,
    max_frames=180,
    highlight_joint="wrist",
)

print("saved:", pose_overlay_mp4)
display_video_in_notebook(pose_overlay_mp4, width=720, muted=True, loop=True)


## Checks｜交給 Day 4-03 前，我會請你們確認

- 骨架在投籃前後大致黏在同一位投籃者身上，沒有跳到場上其他人。
- `wrist` 點落在當下手腕；沒有把手腕標記誤認成籃球。
- `shot_score` 高峰與投籃動作大致重疊，角度曲線沒有大量不合理跳動。
- `d4_02_pose_angles.csv` 已經寫入 `assets/results/`，而且來源影片和 Day 4-01 相同。

完成後，我們會在 Day 4-03 把投籃候選、投籃者手腕與所有球候選接在一起，建立真正的事件球軌跡。
